In [1]:
from typing import Optional, Tuple
import torch
import torch.nn as nn

In [ ]:
class SiglipVisionConfig:
    # Holds all hyperparameters for the SigLIP Vision Transformer model
    def __init__(
        self,
        hidden_size=768,              # size of hidden embeddings
        intermediate_size=3072,       # size of MLP layer inside transformer block
        num_hidden_layers=12,         # number of transformer encoder layers
        num_attention_head=12,        # number of attention heads
        num_channels=3,               # input channels (3 for RGB)
        image_size=224,               # input image resolution
        patch_size=16,                # patch size for splitting image
        layer_norm_eps=1e-6,          # epsilon for layer norm stability
        attention_dropout=0.0,        # dropout in attention
        num_image_tokens: int = None, # optional, total number of image tokens
        **kwargs                      # extra unused args
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_head = num_attention_head
        self.num_channels = num_channels
        self.image_size = image_size
        self.patch_size = patch_size
        self.layer_norm_eps = layer_norm_eps
        self.attention_dropout = attention_dropout
        self.num_image_tokens = num_image_tokens

In [3]:
class SiglipVisionEmbeddings(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size  # embedding dimension for patches
        self.image_size = config.image_size  # input image size
        self.patch_size = config.patch_size  # patch size for splitting image
        
        self.patch_embedding = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=self.embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
            padding="valid", # This indicates no padding is added
        )
        
        self.num_patches = (self.image_size // self.patch_size) ** 2  # total number of patches
        self.num_positions = self.num_patches  # positions = number of patches
        self.position_embedding = nn.Embedding(self.num_positions, self.embed_dim)  # learnable positional embeddings
        
        self.register_buffer(
            "position_ids",
            torch.arange(self.num_positions).expand((1,-1)),  # position indices for each patch
            persistant=False,  # non-trainable buffer
        )
        
    def forward(self, pixel_values: torch.FloatTensor) -> torch.Tensor:
        _, _, height, width = pixel_values.shape # [Batch_Size, Channels, Height, Width]
        # Convolve the `patch_size` kernel over the image, with no overlapping patches since the stride is equal to the kernel size
        # The output of the convolution will have shape [Batch_Size, Embed_Dim, Num_Patches_H, Num_Patches_W]
        # Where Num_Patches_H = height // patch_size and Num_Patches_W = width // patch_size
        patch_embeds = self.patch_embedding(pixel_values)
        # [Batch_Size, Embed_Dim, Num_Patches_H, Num_Patches_W] -> [Batch_Size, Embed_Dim, Num_Patches]
        # where Num_patches = Num_Patches_H * Num_Patches_W
        embeddings = patch_embeds.flatten(2)  # flatten height & width into patch dimension
        # [Batch_Size, Embed_Dim, Num_Patches] -> [Batch_Size, Num_Patches, Embed_Dim]
        embeddings = embeddings.transpose(1, 2)  # switch dimensions to match transformer input
        # Add position embeddings to each patch. Each positional encoding is a vector of size [Embed_Dim]
        embeddings = embeddings + self.position_embedding(self.position_ids)  # add positional info
        # [Batch_Size, Num_Patches, Embed_Dim]
        return embeddings


In [ ]:
class SiglipVisionTransformer(nn.Module):
    # Main Vision Transformer model using SigLIP config
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        embed_dim = config.hidden_size  # embedding dimension for all layers
        
        self.embeddings = SiglipVisionEmbeddings(config)  # converts image to patch embeddings
        self.encoder = SiglipEncoder(config)               # stack of transformer blocks
        self.post_layernorm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)  # final layer norm
        
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # pixel_values: [Batch_Size, Channels, Height, Width] -> [Batch_Size, Num_Patches, Embed_Dim]
        hidden_states = self.embeddings(pixel_values)        # split image into patches + embed
        last_hidden_state = self.encoder(inputs_embeds=hidden_states)  # transformer blocks
        last_hidden_state = self.post_layernorm(last_hidden_state)    # layer norm at the end
        return last_hidden_state

In [ ]:
class SiglipVisionModel(nn.Module):
    # Wrapper model that runs the Vision Transformer
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.vision_model = SiglipVisionTransformer(config)  # core transformer backbone
        
    def forward(self, pixel_values) -> Tuple:
        # image tensor -> sequence of patch embeddings processed by transformer
        return self.vision_model(pixel_values=pixel_values)
